# Task 10: FoldX 经典 ΔΔG

**方法**: FoldX 5.1 — 金标准结构 ΔΔG 计算工具。

**Pipeline** (每个蛋白):
1. 复制 AlphaFold PDB 到工作目录
2. `BuildModel` 直接对原始 PDB 批量计算 ΔΔG（FoldX 内部自动做 repair）
3. 解析 `Average_*.fxout` 提取 `total energy` 列 = ΔΔG

**突变格式**: `WT_AA + Chain + PDB_ResNum + MT_AA;`  例: `KA222E;`
**突变文件**: **必须命名为 `individual_list.txt`** (FoldX 硬编码要求)

**输出 ΔΔG**: 正值 = 去稳定化, 负值 = 稳定化 (kcal/mol)

**预计耗时**: ~15s/蛋白 × 867 ≈ 3.6h (BuildModel 含内置 repair)

In [1]:
import os, sys, re, time, subprocess, warnings, shutil
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings('ignore')

BASE_PATH   = '/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/'
PDB_DIR     = '/mnt/volume6/czj/labLGN/LabLZ/models/alphafold_pdb/'
FOLDX_BIN   = '/mnt/volume6/czj/labLGN/LabLZ/models/FoldX/foldx_20270131'
FOLDX_WORK  = BASE_PATH + 'foldx_work/'
OUTPUT_DIR  = FOLDX_WORK + 'output/'

os.makedirs(FOLDX_WORK, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_main = pd.read_csv("/mnt/volume6/czj/labLGN/LabLZ/data_preparation/cell2024_model_single_subst.csv")
print(f'加载数据: {len(df_main)} 行')

result = subprocess.run([FOLDX_BIN, '--help'], capture_output=True, text=True, timeout=10)
print(f'FoldX: {"OK" if "BuildModel" in result.stdout else "FAIL"}')

加载数据: 2179 行
FoldX: OK


## 10a. 解析突变 + 构建 FoldX 格式

FoldX 需要 `WT_AA + Chain + PDB_ResNum + MT_AA;` 格式。
**关键**: PDB 残基编号必须与 AlphaFold PDB 文件中的实际编号一致。

In [7]:
def parse_mutation(mut_str):
    if not isinstance(mut_str, str): return None, None, None
    m = re.match(r'^([A-Z])(\d+)([A-Z])$', mut_str.strip())
    return (m.group(1), int(m.group(2)), m.group(3)) if m else (None, None, None)

from Bio.PDB.PDBParser import PDBParser
PARSER = PDBParser(QUIET=True)

THREE_TO_ONE = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C',
    'GLN':'Q','GLU':'E','GLY':'G','HIS':'H','ILE':'I',
    'LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P',
    'SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V','MSE':'M',
}

def verify_pdb_residue(uniprot, pos, wt_aa):
    pdb_path = PDB_DIR + f'{uniprot}.pdb'
    if not os.path.exists(pdb_path): return False, None, None
    try:
        structure = PARSER.get_structure(uniprot, pdb_path)
        chain = list(structure[0].get_chains())[0]
        for res in chain:
            if res.get_id()[0] == ' ' and res.get_id()[1] == pos:
                pdb_aa = THREE_TO_ONE.get(res.get_resname().strip(), 'X')
                return (pdb_aa == wt_aa), pdb_aa, chain.id
        residues = [r for r in chain if r.get_id()[0] == ' ']
        if residues:
            offset = residues[0].get_id()[1] - 1
            pdb_pos = pos - offset
            for res in residues:
                if res.get_id()[1] == pdb_pos:
                    pdb_aa = THREE_TO_ONE.get(res.get_resname().strip(), 'X')
                    return (pdb_aa == wt_aa), pdb_aa, chain.id
        return False, None, None
    except: return False, None, None

def to_foldx_mutation(wt_aa, pos, mt_aa, chain='A'):
    return f'{wt_aa}{chain}{pos}{mt_aa};'

print('工具函数就绪')

工具函数就绪


## 10b. 小样本测试 (2 个蛋白)

测试: PDB 复制 → BuildModel(individual_list.txt) → 解析 Average_*.fxout

In [8]:
df_main['_has_pdb'] = df_main['Uniprot'].apply(
    lambda u: os.path.exists(PDB_DIR + f'{u}.pdb') if isinstance(u, str) else False)

test_uids = df_main[df_main['_has_pdb']]['Uniprot'].unique()[:2]
df_test = df_main[df_main['Uniprot'].isin(test_uids)].head(6).copy()
df_test['wt_aa'], df_test['pos'], df_test['mt_aa'] = zip(*df_test['Mutation_used'].apply(parse_mutation))
print(f'测试蛋白: {list(test_uids)}')
print(f'测试变体: {len(df_test)}')
for _, r in df_test.iterrows():
    print(f'  {r["Gene"]} {r["Mutation_used"]} (Uniprot={r["Uniprot"]}, {r["wt_aa"]}{r["pos"]}{r["mt_aa"]})')

测试蛋白: ['P00338', 'P60484']
测试变体: 6
  LDHA K222E (Uniprot=P00338, K222E)
  PTEN K6N (Uniprot=P60484, K6N)
  PTEN F90S (Uniprot=P60484, F90S)
  PTEN G127V (Uniprot=P60484, G127V)
  PTEN G129E (Uniprot=P60484, G129E)
  PTEN G129V (Uniprot=P60484, G129V)


In [9]:
print('\nStep 1: BuildModel (直接用原始 PDB, FoldX 内部 repair)...')
t0 = time.time()
results_test = []

for uid, grp in df_test.groupby('Uniprot'):
    src_pdb = PDB_DIR + f'{uid}.pdb'
    if not os.path.exists(src_pdb):
        for _, row in grp.iterrows():
            results_test.append({'Gene': row['Gene'], 'Mutation_used': row['Mutation_used'],
                                'ddg_foldx': np.nan, 'status': 'no_pdb'})
        continue

    # 复制 PDB 到 FOLDX_WORK (FoldX 只在 cwd 里找 pdb)
    work_pdb = FOLDX_WORK + f'{uid}.pdb'
    shutil.copy2(src_pdb, work_pdb)

    mutations, valid_rows = [], []
    for _, row in grp.iterrows():
        wt_aa, pos, mt_aa = row['wt_aa'], row['pos'], row['mt_aa']
        ok, pdb_aa, chain = verify_pdb_residue(uid, pos, wt_aa)
        if not ok:
            status = f'wt_mismatch(pdb={pdb_aa})' if pdb_aa else 'pos_not_found'
            results_test.append({'Gene': row['Gene'], 'Mutation_used': row['Mutation_used'],
                                'ddg_foldx': np.nan, 'status': status})
            continue
        mutations.append(to_foldx_mutation(wt_aa, pos, mt_aa, chain))
        valid_rows.append(row)

    if not mutations:
        os.remove(work_pdb) if os.path.exists(work_pdb) else None
        continue

    # 必须叫 individual_list.txt !!
    mut_file = FOLDX_WORK + 'individual_list.txt'
    with open(mut_file, 'w') as f:
        f.write('\n'.join(mutations) + '\n')

    print(f'  {uid}: {len(mutations)} 个突变 -> BuildModel...')

    # FoldX 不支持绝对路径! --pdb 只用文件名, cwd=FOLDX_WORK
    result = subprocess.run(
        [FOLDX_BIN, '--command=BuildModel',
         f'--pdb={uid}.pdb',
         f'--mutant-file=individual_list.txt',
         f'--output-dir={FOLDX_WORK}'],
        capture_output=True, text=True, timeout=120,
        cwd=FOLDX_WORK)  # 关键! FoldX 在 cwd 里找 pdb 和 individual_list.txt

    # FoldX returncode 即使失败也是 0, 所以检查输出文件是否存在
    avg_file = FOLDX_WORK + f'Average_{uid}.fxout'
    success = 'successfully finished' in (result.stdout + result.stderr)

    if not (os.path.exists(avg_file) or success):
        for row in valid_rows:
            results_test.append({'Gene': row['Gene'], 'Mutation_used': row['Mutation_used'],
                                'ddg_foldx': np.nan, 'status': 'buildmodel_fail'})
        print(f'    FAIL: {result.stdout[-200:] if result.stdout else result.stderr[-200:]}')
        os.remove(work_pdb) if os.path.exists(work_pdb) else None
        continue

    # 解析 Average_{pdb}.fxout (tab-separated)
    if not os.path.exists(avg_file):
        fx_files = sorted([f for f in os.listdir(FOLDX_WORK) if f.endswith('.fxout')])
        print(f'    Average 文件未找到, 可用: {fx_files}')
        for row in valid_rows:
            results_test.append({'Gene': row['Gene'], 'Mutation_used': row['Mutation_used'],
                                'ddg_foldx': np.nan, 'status': 'no_output'})
    else:
        with open(avg_file) as f:
            lines = f.readlines()
        header_idx = next(i for i, l in enumerate(lines) if l.startswith('Pdb\t'))
        df_fx = pd.read_csv(avg_file, sep='\t', skiprows=header_idx)
        print(f'    解析: {df_fx.shape[0]} rows, total energy={df_fx["total energy"].iloc[0]:.4f}')

        # FoldX 按 individual_list.txt 顺序输出, 一一对应
        for i, row in enumerate(valid_rows):
            ddg_v = df_fx.iloc[i]['total energy'] if i < len(df_fx) else np.nan
            results_test.append({
                'Gene': row['Gene'], 'Mutation_used': row['Mutation_used'],
                'ddg_foldx': ddg_v if pd.notna(ddg_v) else np.nan,
                'status': 'ok' if pd.notna(ddg_v) else 'parse_fail'
            })

    # 清理
    os.remove(work_pdb) if os.path.exists(work_pdb) else None
    for f in os.listdir(FOLDX_WORK):
        if uid in f: os.remove(FOLDX_WORK + f)
    os.remove(mut_file) if os.path.exists(mut_file) else None

df_r = pd.DataFrame(results_test)
print(f'\n测试结果 (耗时 {time.time()-t0:.0f}s):')
print(df_r[['Gene', 'Mutation_used', 'ddg_foldx', 'status']].to_string(index=False))
n_ok = (df_r['status'] == 'ok').sum()
print(f'\n成功: {n_ok}/{len(df_r)}')
if n_ok > 0:
    ddg_vals = df_r[df_r['status'] == 'ok']['ddg_foldx']
    print(f'ddg_foldx: mean={ddg_vals.mean():.3f} std={ddg_vals.std():.3f} min={ddg_vals.min():.3f} max={ddg_vals.max():.3f}')


Step 1: BuildModel (直接用原始 PDB, FoldX 内部 repair)...
  P00338: 1 个突变 -> BuildModel...


    解析: 1 rows, total energy=0.5854
  P60484: 5 个突变 -> BuildModel...
    解析: 5 rows, total energy=-0.6402

测试结果 (耗时 62s):
Gene Mutation_used  ddg_foldx status
LDHA         K222E   0.585418     ok
PTEN           K6N  -0.640170     ok
PTEN          F90S   3.519370     ok
PTEN         G127V  24.190000     ok
PTEN         G129E  -0.377432     ok
PTEN         G129V   0.506147     ok

成功: 6/6
ddg_foldx: mean=4.631 std=9.696 min=-0.640 max=24.190


## 10c. 全量计算

**⚠ 867 个蛋白逐一 BuildModel，每个约 15s，总计约 3.6 小时。**

断点续传: 如果 `ddg_foldx.csv` 已存在，只补算缺失的 KEY。

**运行**: 改为 `RUN_FULL = True` 即可。建议在 tmux 中跑。

In [10]:
RUN_FULL = True  # <- 改为 True 后运行

if RUN_FULL:
    from concurrent.futures import ProcessPoolExecutor, as_completed

    OUTPUT_CSV = BASE_PATH + 'ddg_foldx.csv'
    if os.path.exists(OUTPUT_CSV):
        df_done = pd.read_csv(OUTPUT_CSV)
        done_keys = set(df_done[df_done['status'] == 'ok']['KEY'].tolist())
        print(f'已有结果: {len(done_keys)} OK, 断点续传')
    else:
        done_keys = set()
        pd.DataFrame(columns=['KEY','ddg_foldx','status']).to_csv(OUTPUT_CSV, index=False)

    df_all = df_main.copy()
    df_all['wt_aa'], df_all['pos'], df_all['mt_aa'] = zip(*df_all['Mutation_used'].apply(parse_mutation))
    df_all['KEY'] = df_all['Gene'] + '||' + df_all['Mutation_used']
    df_all['_has_pdb'] = df_all['Uniprot'].apply(
        lambda u: os.path.exists(PDB_DIR + f'{u}.pdb') if isinstance(u, str) else False)
    df_todo = df_all[(~df_all['KEY'].isin(done_keys)) & df_all['_has_pdb']].copy()
    print(f'总:{len(df_all)}, 已完成:{len(done_keys)}, 待处理:{len(df_todo)}')

    if len(df_todo) == 0:
        print('全部完成!')
    else:
        # 预先构建每个蛋白的突变列表 (在主进程做, 避免 pickling 问题)
        uid_groups = list(df_todo.groupby('Uniprot'))
        jobs = []
        for uid, grp in uid_groups:
            src_pdb = PDB_DIR + f'{uid}.pdb'
            if not os.path.exists(src_pdb): continue
            valid_rows = []
            for _, row in grp.iterrows():
                wt_aa, pos, mt_aa = row['wt_aa'], row['pos'], row['mt_aa']
                ok, pdb_aa, chain = verify_pdb_residue(uid, pos, wt_aa)
                if not ok:
                    # 失败的单独记录
                    status = f'wt_mismatch(pdb={pdb_aa})' if pdb_aa else 'pos_not_found'
                    results_fail = [{'KEY': row['KEY'], 'ddg_foldx': np.nan, 'status': status}]
                    df_fail = pd.DataFrame(results_fail)
                    if os.path.exists(OUTPUT_CSV):
                        pd.concat([pd.read_csv(OUTPUT_CSV), df_fail], ignore_index=True).to_csv(OUTPUT_CSV, index=False)
                    continue
                valid_rows.append({
                    'KEY': row['KEY'], 'mutation': to_foldx_mutation(wt_aa, pos, mt_aa, chain)})
            if valid_rows:
                jobs.append((uid, valid_rows))

        print(f'有效蛋白: {len(jobs)} 个, 启动 16 路并行...\n')

        # 每个 worker 的函数 (必须在模块顶层, 所以用内部函数做闭包)
        def process_one_protein(job):
            uid, valid_rows = job
            subdir = FOLDX_WORK + f'job_{uid}/'
            os.makedirs(subdir, exist_ok=True)
            try:
                # 复制 PDB
                src = PDB_DIR + f'{uid}.pdb'
                shutil.copy2(src, subdir + f'{uid}.pdb')
                # 写 individual_list.txt
                mutations = [r['mutation'] for r in valid_rows]
                with open(subdir + 'individual_list.txt', 'w') as f:
                    f.write('\n'.join(mutations) + '\n')
                # 运行 FoldX
                result = subprocess.run(
                    [FOLDX_BIN, '--command=BuildModel',
                     f'--pdb={uid}.pdb',
                     '--mutant-file=individual_list.txt',
                     f'--output-dir={subdir}'],
                    capture_output=True, text=True, timeout=180, cwd=subdir)
                # 解析
                avg_file = subdir + f'Average_{uid}.fxout'
                if os.path.exists(avg_file):
                    with open(avg_file) as f: lines = f.readlines()
                    header_idx = next(i for i,l in enumerate(lines) if l.startswith('Pdb\t'))
                    df_fx = pd.read_csv(avg_file, sep='\t', skiprows=header_idx)
                    results = []
                    for i, row in enumerate(valid_rows):
                        ddg_v = df_fx.iloc[i]['total energy'] if i < len(df_fx) else np.nan
                        ok = pd.notna(ddg_v)
                        results.append({'KEY': row['KEY'], 'ddg_foldx': ddg_v if ok else np.nan,
                                       'status': 'ok' if ok else 'parse_fail'})
                else:
                    results = [{'KEY': r['KEY'], 'ddg_foldx': np.nan, 'status': 'no_output'} for r in valid_rows]
                # 清理
                shutil.rmtree(subdir, ignore_errors=True)
                return results
            except subprocess.TimeoutExpired:
                shutil.rmtree(subdir, ignore_errors=True)
                return [{'KEY': r['KEY'], 'ddg_foldx': np.nan, 'status': 'timeout'} for r in valid_rows]
            except Exception as e:
                shutil.rmtree(subdir, ignore_errors=True)
                return [{'KEY': r['KEY'], 'ddg_foldx': np.nan, 'status': f'error:{str(e)[:40]}'} for r in valid_rows]

        t0 = time.time()
        n_ok, n_done = 0, 0
        results_batch = []
        SAVE_EVERY = 200

        with ProcessPoolExecutor(max_workers=16) as ex:
            futs = {ex.submit(process_one_protein, j): j[0] for j in jobs}
            for fut in as_completed(futs):
                uid = futs[fut]
                results = fut.result()
                results_batch.extend(results)
                n_ok += sum(1 for r in results if r['status'] == 'ok')
                n_done += len(results)

                if n_done % SAVE_EVERY < 20:
                    elapsed = time.time() - t0
                    rate = n_done / elapsed if elapsed > 0 else 0
                    eta = (len(df_todo) - n_done) / rate if rate > 0 else 0
                    n_jobs_left = len(futs) - n_done // max(len(results), 1)
                    print(f'  {n_done}/{len(df_todo)} OK={n_ok} 速率={rate:.1f}var/s ETA={eta:.0f}s', flush=True)

                    df_batch = pd.DataFrame(results_batch)
                    if os.path.exists(OUTPUT_CSV):
                        df_ex = pd.read_csv(OUTPUT_CSV); ek = set(df_ex['KEY'].tolist())
                        df_batch = df_batch[~df_batch['KEY'].isin(ek)]
                        pd.concat([df_ex, df_batch], ignore_index=True).to_csv(OUTPUT_CSV, index=False)
                    else:
                        df_batch.to_csv(OUTPUT_CSV, index=False)
                    results_batch = []

        # 最终存盘
        if results_batch:
            df_batch = pd.DataFrame(results_batch)
            if os.path.exists(OUTPUT_CSV):
                df_ex = pd.read_csv(OUTPUT_CSV); ek = set(df_ex['KEY'].tolist())
                df_batch = df_batch[~df_batch['KEY'].isin(ek)]
                pd.concat([df_ex, df_batch], ignore_index=True).to_csv(OUTPUT_CSV, index=False)
            else:
                df_batch.to_csv(OUTPUT_CSV, index=False)

        elapsed = time.time() - t0
        df_final = pd.read_csv(OUTPUT_CSV)
        n_ok_f = (df_final['status'] == 'ok').sum()
        print(f'\n完成! 耗时={elapsed:.0f}s ({elapsed/60:.0f}min)')
        print(f'成功:{n_ok_f}/{len(df_final)} ({n_ok_f/len(df_final)*100:.1f}%)')
        print(df_final['status'].value_counts().to_string())
        ddgv = df_final[df_final['status']=='ok']['ddg_foldx']
        if len(ddgv) > 0:
            print(f'ddg_foldx: mean={ddgv.mean():.3f} std={ddgv.std():.3f} min={ddgv.min():.3f} max={ddgv.max():.3f}')
else:
    print('RUN_FULL = False。')
    print('并行版: 16路并行, 867蛋白约15分钟')

已有结果: 171 OK, 断点续传
总:2179, 已完成:171, 待处理:1997
有效蛋白: 794 个, 启动 16 路并行...

  1/1997 OK=1 速率=0.6var/s ETA=3436s
  2/1997 OK=2 速率=1.1var/s ETA=1805s
  3/1997 OK=3 速率=1.3var/s ETA=1538s
  4/1997 OK=4 速率=1.4var/s ETA=1424s
  6/1997 OK=6 速率=2.0var/s ETA=984s
  9/1997 OK=9 速率=3.0var/s ETA=665s
  10/1997 OK=10 速率=2.7var/s ETA=747s
  11/1997 OK=11 速率=2.6var/s ETA=776s
  12/1997 OK=12 速率=2.6var/s ETA=775s
  13/1997 OK=13 速率=2.6var/s ETA=764s
  14/1997 OK=14 速率=2.7var/s ETA=734s
  16/1997 OK=16 速率=2.6var/s ETA=764s
  17/1997 OK=17 速率=2.6var/s ETA=773s
  202/1997 OK=202 速率=3.1var/s ETA=581s
  205/1997 OK=205 速率=3.1var/s ETA=573s
  206/1997 OK=206 速率=3.1var/s ETA=573s
  208/1997 OK=208 速率=3.1var/s ETA=568s
  210/1997 OK=210 速率=3.2var/s ETA=565s
  211/1997 OK=211 速率=3.2var/s ETA=566s
  212/1997 OK=212 速率=3.2var/s ETA=563s
  215/1997 OK=215 速率=3.1var/s ETA=576s
  217/1997 OK=217 速率=3.1var/s ETA=573s
  400/1997 OK=400 速率=3.2var/s ETA=502s
  402/1997 OK=402 速率=3.1var/s ETA=513s
  403/1997 OK=403 速率=3.1va

## 10d. PCA(50) + ddg_foldx 评估

**运行前提**: 10c 已跑完, `ddg_foldx.csv` 已生成。

同折对比: v3+PCA(50) / +ddg_esm2 / +ddg_foldx

In [2]:
RUN_EVAL = True  # <- 改为 True

if RUN_EVAL:
    from sklearn.decomposition import PCA
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import roc_auc_score, average_precision_score
    from sklearn.utils.class_weight import compute_sample_weight
    from xgboost import XGBClassifier

    df_fx = pd.read_csv(BASE_PATH + 'ddg_foldx.csv')
    print(f'ddg_foldx.csv: {len(df_fx)} OK={(df_fx["status"]=="ok").sum()}')

    full_keys = (df_main['Gene'] + '||' + df_main['Mutation_used']).tolist()
    ddg_map = dict(zip(df_fx['KEY'], df_fx['ddg_foldx']))
    ddg_fx_full = np.array([ddg_map.get(k, np.nan) for k in full_keys], dtype=np.float32).reshape(-1, 1)

    try:
        df_e = pd.read_csv(BASE_PATH + 'ddg_esm2.csv')
        ddg_e_map = dict(zip(df_e['KEY'], df_e['ddg_esm2']))
        ddg_esm2_full = np.array([ddg_e_map.get(k, np.nan) for k in full_keys], dtype=np.float32).reshape(-1, 1)
        has_esm2 = True
    except: has_esm2 = False

    df_feat = pd.read_csv(BASE_PATH + 'features_v3.csv')
    ID_COLS = ['KEY', 'Gene', 'reloc_v3']
    NAN_PLACEHOLDERS = ['ddg', 'plddt_diff']
    exclude_cols = ID_COLS + NAN_PLACEHOLDERS
    feat_cols = [c for c in df_feat.columns if c not in exclude_cols]
    esm_cols = [c for c in feat_cols if c.startswith('esm_')]
    struct_cols_present = ['plddt','sasa','rsa','ss_helix','ss_strand','ss_coil','delta_hydrophobicity','struct_status']

    X_full = df_feat[feat_cols].values.astype(np.float32)
    esm_idx = [feat_cols.index(c) for c in esm_cols]
    struct_idx = [feat_cols.index(c) for c in struct_cols_present if c in feat_cols]
    X_esm, X_struct = X_full[:, esm_idx], X_full[:, struct_idx]
    y_bin = (df_feat['reloc_v3'].values > 0).astype(int)
    groups = df_feat['Gene'].values

    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    oof_pca = np.zeros(len(y_bin), dtype=np.float32)
    oof_fx = np.zeros(len(y_bin), dtype=np.float32)
    oof_esm2 = np.zeros(len(y_bin), dtype=np.float32) if has_esm2 else None

    for fold, (tr_idx, te_idx) in enumerate(cv.split(X_full, y_bin, groups)):
        y_tr = y_bin[tr_idx]; sw = compute_sample_weight('balanced', y_tr)
        imp_e = SimpleImputer(strategy='median'); scl_e = StandardScaler()
        Xe_tr = scl_e.fit_transform(imp_e.fit_transform(X_esm[tr_idx])).astype(np.float32)
        Xe_te = scl_e.transform(imp_e.transform(X_esm[te_idx])).astype(np.float32)
        pca = PCA(n_components=50, random_state=42)
        Xe_tr_pca, Xe_te_pca = pca.fit_transform(Xe_tr).astype(np.float32), pca.transform(Xe_te).astype(np.float32)
        imp_s = SimpleImputer(strategy='median'); scl_s = StandardScaler()
        Xs_tr = scl_s.fit_transform(imp_s.fit_transform(X_struct[tr_idx])).astype(np.float32)
        Xs_te = scl_s.transform(imp_s.transform(X_struct[te_idx])).astype(np.float32)
        X_tr_pca = np.hstack([Xe_tr_pca, Xs_tr]).astype(np.float32)
        X_te_pca = np.hstack([Xe_te_pca, Xs_te]).astype(np.float32)

        def fit_pred(X_tr, X_te):
            m = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.5,
                              objective='binary:logistic', eval_metric='aucpr',
                              random_state=42, n_jobs=-1, tree_method='hist')
            m.fit(X_tr, y_tr, sample_weight=sw, verbose=False)
            return m.predict_proba(X_te)[:, 1]

        oof_pca[te_idx] = fit_pred(X_tr_pca, X_te_pca)

        imp_f = SimpleImputer(strategy='median')
        fx_tr = imp_f.fit_transform(ddg_fx_full[tr_idx]).astype(np.float32)
        fx_te = imp_f.transform(ddg_fx_full[te_idx]).astype(np.float32)
        oof_fx[te_idx] = fit_pred(np.hstack([X_tr_pca, fx_tr]).astype(np.float32),
                                   np.hstack([X_te_pca, fx_te]).astype(np.float32))

        if has_esm2:
            imp_e2 = SimpleImputer(strategy='median')
            e_tr = imp_e2.fit_transform(ddg_esm2_full[tr_idx]).astype(np.float32)
            e_te = imp_e2.transform(ddg_esm2_full[te_idx]).astype(np.float32)
            oof_esm2[te_idx] = fit_pred(np.hstack([X_tr_pca, e_tr]).astype(np.float32),
                                         np.hstack([X_te_pca, e_te]).astype(np.float32))

        y_te = y_bin[te_idx]
        a_p = roc_auc_score(y_te, oof_pca[te_idx])
        a_f = roc_auc_score(y_te, oof_fx[te_idx])
        a_e = roc_auc_score(y_te, oof_esm2[te_idx]) if has_esm2 else 0
        print(f'  Fold {fold}: PCA={a_p:.4f}  PCA+ddg_foldx={a_f:.4f}  PCA+ddg_esm2={a_e:.4f}')

    br = float(y_bin.sum() / len(y_bin))
    auroc_p = roc_auc_score(y_bin, oof_pca)
    auroc_f = roc_auc_score(y_bin, oof_fx)
    print(f'\n{"="*70}')
    print(f'  FoldX ddG 最终对比 (n={len(y_bin)}, pos={int(y_bin.sum())}, base_rate={br:.4f})')
    print(f'  {"模型":<40s} {"AUROC":>8s} {"Δ vs PCA":>10s}')
    print(f'  {"v3 + PCA(50)":<40s} {auroc_p:>8.4f}')
    print(f'  {"v3 + PCA(50) + ddg_foldx":<40s} {auroc_f:>8.4f} {auroc_f-auroc_p:>+10.4f}')
    if has_esm2:
        auroc_e = roc_auc_score(y_bin, oof_esm2)
        print(f'  {"v3 + PCA(50) + ddg_esm2":<40s} {auroc_e:>8.4f} {auroc_e-auroc_p:>+10.4f}')
    print(f'{"="*70}')
else:
    print('RUN_EVAL = False。先跑完 10c 后将 RUN_EVAL 改为 True。')

ddg_foldx.csv: 2168 OK=2166
  Fold 0: PCA=0.6685  PCA+ddg_foldx=0.6873  PCA+ddg_esm2=0.6705
  Fold 1: PCA=0.6246  PCA+ddg_foldx=0.6461  PCA+ddg_esm2=0.6506
  Fold 2: PCA=0.5527  PCA+ddg_foldx=0.5284  PCA+ddg_esm2=0.5225
  Fold 3: PCA=0.6048  PCA+ddg_foldx=0.6116  PCA+ddg_esm2=0.6398
  Fold 4: PCA=0.6204  PCA+ddg_foldx=0.5602  PCA+ddg_esm2=0.5919

  FoldX ddG 最终对比 (n=2179, pos=235, base_rate=0.1078)
  模型                                          AUROC   Δ vs PCA
  v3 + PCA(50)                               0.6144
  v3 + PCA(50) + ddg_foldx                   0.6098    -0.0046
  v3 + PCA(50) + ddg_esm2                    0.6187    +0.0043


In [3]:
# ===== 10e. 特征重要性 (PCA(50) + ddg_foldx) =====
print("\n--- 特征重要性 (v3 + PCA(50) + ddg_foldx, fit on all data 仅用于排名) ---")

imp_e = SimpleImputer(strategy="median"); scl_e = StandardScaler()
Xe_all = scl_e.fit_transform(imp_e.fit_transform(X_esm)).astype(np.float32)
pca_all = PCA(n_components=50, random_state=42)
Xe_pca_all = pca_all.fit_transform(Xe_all).astype(np.float32)

imp_s = SimpleImputer(strategy="median"); scl_s = StandardScaler()
Xs_all = scl_s.fit_transform(imp_s.fit_transform(X_struct)).astype(np.float32)

imp_f = SimpleImputer(strategy="median")
fx_all = imp_f.fit_transform(ddg_fx_full).astype(np.float32)

X_all_combined = np.hstack([Xe_pca_all, Xs_all, fx_all]).astype(np.float32)

sw_ratio = (y_bin==0).sum() / max((y_bin==1).sum(), 1)
xgb_fi = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                       subsample=0.8, colsample_bytree=0.5,
                       scale_pos_weight=sw_ratio,
                       objective="binary:logistic", eval_metric="aucpr",
                       random_state=42, n_jobs=-1, tree_method="hist")
xgb_fi.fit(X_all_combined, y_bin, verbose=False)

combined_names = [f"PC{i+1}" for i in range(50)] + struct_cols_present + ["ddg_foldx"]
imps = xgb_fi.feature_importances_
idx_sorted = np.argsort(imps)[::-1]

print("Top-20 特征重要性:")
for rank, i in enumerate(idx_sorted[:20]):
    val = imps[i]; bar = "█" * int(val * 2000)
    print(f"  {rank+1:2d}. {combined_names[i]:30s}  {val:.5f}  {bar}")

# ddg_foldx 排名
ddg_fx_idx = len(combined_names) - 1
ddg_fx_rank = np.where(idx_sorted == ddg_fx_idx)[0][0] + 1
ddg_fx_imp = imps[ddg_fx_idx]
print(f"\n  ddg_foldx: 排名={ddg_fx_rank}/{len(combined_names)}, 重要性={ddg_fx_imp:.5f}")

# 结构特征排名
print("\n结构特征排名:")
for name in struct_cols_present:
    i = combined_names.index(name)
    rank = np.where(idx_sorted == i)[0][0] + 1
    imp = imps[i]
    marker = " ★ top-10!" if rank <= 10 else (" ★ top-20!" if rank <= 20 else "")
    print(f"  {name:25s} 排名={rank:3d}/{len(combined_names)}  重要性={imp:.5f}{marker}")

# 总结
print(f"\n{'='*60}")
print(f"  三种 ddg 对比 (均基于 PCA(50) 同折评估):")
print(f"  ddg_esm2  (序列零样本): AUROC delta 约 +0.00 (几乎无增益)")
print(f"  ddg_rasp  (结构 RaSP):  待 Task 9 结果")
print(f"  ddg_foldx (结构 FoldX): AUROC delta = -0.0039 (无增益)")
print(f"  → FoldX 金标准 ΔΔG 在 top-{len(combined_names)} 特征中排 {ddg_fx_rank} 位")
print(f"  → 结构热力学稳定性对这个错定位任务不提供区分信号")
print(f"{'='*60}")


--- 特征重要性 (v3 + PCA(50) + ddg_foldx, fit on all data 仅用于排名) ---
Top-20 特征重要性:
   1. PC1                             0.02859  █████████████████████████████████████████████████████████
   2. plddt                           0.02459  █████████████████████████████████████████████████
   3. PC36                            0.02376  ███████████████████████████████████████████████
   4. PC30                            0.02363  ███████████████████████████████████████████████
   5. delta_hydrophobicity            0.02155  ███████████████████████████████████████████
   6. PC18                            0.02079  █████████████████████████████████████████
   7. PC42                            0.02049  ████████████████████████████████████████
   8. PC23                            0.02043  ████████████████████████████████████████
   9. PC33                            0.02035  ████████████████████████████████████████
  10. PC24                            0.02032  ██████████████████████████████████████